In [2]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3

# %matplotlib qt
%matplotlib inline
import matplotlib.pyplot as plt

from pathlib import Path
import spikeinterface.extractors as se
import spikeinterface.full as si
from neuropy.core.session.init_from_raw_data import windows_to_wsl_path_if_needed, find_first_file_rglob

from spikeinterface_pipeline import BapunSessionConfig, CurationConfig, resolve_session_paths, load_bapun_recording, load_phy_sorting, load_spykingcircus_sorting, run_phy_curation_pipeline, compute_qm_labels


Automatic pdb calling has been turned OFF
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
## Bapun Format:
# basedir = Path('/home/halechr/FastData/Bapun/RatS/Day1Openfield') # Linux
# basedir = Path('/nfs/turbo/umms-kdiba/Data/Bapun/RatS/Day1Openfield') # Greatlakes
# basedir = windows_to_wsl_path_if_needed(Path('W:/Data/Bapun/RatS/Day1Openfield')) # Windows
# basedir = Path("/mnt/w/Data/Bapun/RatS/Day1Openfield") # Apogee WSL
# # n_channels: int = 200
# n_channels: int = 195
# dat_file_sampling_rate: int = 30000
# eeg_sampling_rate: int = 1250
# basename: str = 'RatS-Day1Openfield'
# excluded_data_datetimes = ['2020-11-25_10-24-24', '2020-11-25_15-06-02']


# basedir = Path('/home/halechr/FastData/Bapun/RatS/Day4Openfield') # Linux
# basedir = Path('/nfs/turbo/umms-kdiba/Data/Bapun/RatS/Day4Openfield') # Greatlakes Turbo
basedir = Path('/dev/shm/halechr/Day4Openfield') # Greatlakes SHARED RAM DISK

# basedir = windows_to_wsl_path_if_needed(Path('W:/Data/Bapun/RatS/Day4Openfield')) # Windows
# basedir = Path("/mnt/w/Data/Bapun/RatS/Day4Openfield") # Apogee WSL
n_channels: int = 195
dat_file_sampling_rate: int = 30000
eeg_sampling_rate: int = 1250
basename: str = 'RatS-Day4Openfield'
excluded_data_datetimes = []

# # basedir = Path('/home/halechr/FastData/Bapun/RatS/Day4Openfield') # Linux
# # basedir = Path('/nfs/turbo/umms-kdiba/Bapun/RatS/Day5TwoNovel') # Greatlakes Turbo
# basedir = Path('/scratch/kdiba_root/kdiba99/halechr/Data/Bapun/RatS/Day5TwoNovel') # Greatlakes

# # basedir = windows_to_wsl_path_if_needed(Path('W:/Data/Bapun/RatS/Day4Openfield')) # Windows
# # basedir = Path("/mnt/w/Data/Bapun/RatS/Day4Openfield") # Apogee WSL
# n_channels: int = 195
# dat_file_sampling_rate: int = 30000
# eeg_sampling_rate: int = 1250
# basename: str = 'RatS-Day5TwoNovel-2020-12-04_07-55-09'
# excluded_data_datetimes = []


In [6]:
session = BapunSessionConfig(basedir=basedir, basename=basename, n_channels=n_channels, dat_file_sampling_rate=dat_file_sampling_rate, eeg_sampling_rate=eeg_sampling_rate, excluded_data_datetimes=excluded_data_datetimes)
paths = resolve_session_paths(session)
xml_path = paths.xml_path
concatenated_dat_file = paths.concatenated_dat_file
print(f'xml_path: {xml_path}')
print(f"concatenated_dat_file: {concatenated_dat_file.as_posix()}")
assert concatenated_dat_file.exists(), f"concatenated_dat_file: {concatenated_dat_file} does not exist."
paths

xml_path: /dev/shm/halechr/Day4Openfield/RatS-Day4Openfield.xml
concatenated_dat_file: /dev/shm/halechr/Day4Openfield/spyk-circ/RatS-Day4Openfield.dat


SessionPaths(basedir=PosixPath('/dev/shm/halechr/Day4Openfield'), basename='RatS-Day4Openfield', xml_path=PosixPath('/dev/shm/halechr/Day4Openfield/RatS-Day4Openfield.xml'), concatenated_dat_file=PosixPath('/dev/shm/halechr/Day4Openfield/spyk-circ/RatS-Day4Openfield.dat'), prb_path=PosixPath('/dev/shm/halechr/Day4Openfield/spyk-circ/RatS-Day4Openfield.prb'), spiking_circus_dir=PosixPath('/dev/shm/halechr/Day4Openfield/spyk-circ/RatS-Day4Openfield'), phy_gui_dir=PosixPath('/dev/shm/halechr/Day4Openfield/spyk-circ/RatS-Day4Openfield/RatS-Day4Openfield-merged.GUI'), phy_sorting_analyzer_folder=PosixPath('/dev/shm/halechr/Day4Openfield/spyk-circ/RatS-Day4Openfield-phy-sorting_analyzer'), curation_review_path=PosixPath('/dev/shm/halechr/Day4Openfield/spyk-circ/RatS-Day4Openfield-curation_review.csv'))

In [7]:
spiking_circus_loaded = load_spykingcircus_sorting(paths)
spiking_circus_loaded


SpykingCircusSortingExtractor: 258 units - 1 segments - 30.0kHz

In [8]:
phy_gui_dir = paths.phy_gui_dir
assert phy_gui_dir.exists(), f"phy_gui_dir: {phy_gui_dir} does not exist!"


In [9]:
sorting = load_phy_sorting(paths)
sorting


PhySortingExtractor: 257 units - 1 segments - 30.0kHz

In [ ]:
# spiking_circus_loaded

si.create_sorting_analyzer()

In [ ]:
n_channels

In [10]:
recording, recording_filtered = load_bapun_recording(session, paths)
recording


ChannelSliceRecording: 175 channels - 30.0kHz - 1 segments - 261,680,640 samples 
                       8,722.69s (2.42 hours) - int16 dtype - 85.30 GiB

In [32]:
recording_filtered


BandpassFilterRecording: 176 channels - 30.0kHz - 1 segments - 125,724,262 samples 
                         4,190.81s (1.16 hours) - int16 dtype - 41.22 GiB

In [11]:
sorting = spiking_circus_loaded

job_kwargs = dict(n_jobs=-1, progress_bar=True, chunk_duration="1s")

sorting_analyzer_folder: Path = windows_to_wsl_path_if_needed(basedir.joinpath("spyk-circ", f"{basename}-sorting_analyzer").resolve())
sorting_analyzer = si.create_sorting_analyzer(sorting, recording_filtered, format="binary_folder", folder=sorting_analyzer_folder.as_posix())
sorting_analyzer.compute("random_spikes", method="uniform", max_spikes_per_unit=500)
sorting_analyzer.compute("waveforms", **job_kwargs)
sorting_analyzer.compute("templates", **job_kwargs)
sorting_analyzer.compute("noise_levels")
sorting_analyzer.compute("unit_locations", method="monopolar_triangulation")
sorting_analyzer.compute("isi_histograms")
sorting_analyzer.compute("correlograms", window_ms=100, bin_ms=5.)
sorting_analyzer.compute("principal_components", n_components=3, mode='by_channel_global', whiten=True, **job_kwargs)
sorting_analyzer.compute("quality_metrics", metric_names=["snr", "firing_rate"])
sorting_analyzer.compute("template_similarity")
sorting_analyzer.compute("spike_amplitudes", **job_kwargs)

estimate_sparsity (no parallelization):   0%|          | 0/8723 [00:00<?, ?it/s]

compute_waveforms (workers: 36 processes fork):   0%|          | 0/8723 [00:00<?, ?it/s]

noise_level (no parallelization):   0%|          | 0/20 [00:00<?, ?it/s]

/nfs/turbo/umms-kdiba/Bapun/RatS/spikeinterface/src/spikeinterface/postprocessing/localization_tools.py:341: RuntimeWarning: invalid value encountered in divide
  com = np.sum(wf_data[:, np.newaxis] * local_contact_locations, axis=0) / np.sum(wf_data)
/nfs/turbo/umms-kdiba/Bapun/RatS/spikeinterface/src/spikeinterface/postprocessing/localization_tools.py:367: UserWarning: scipy.optimize.least_squares error: Each lower bound must be strictly less than each upper bound.
  warnings.warn(f"scipy.optimize.least_squares error: {e}")


Fitting PCA:   0%|          | 0/258 [00:00<?, ?it/s]

Projecting waveforms:   0%|          | 0/258 [00:00<?, ?it/s]

/nfs/turbo/umms-kdiba/Bapun/RatS/spikeinterface/src/spikeinterface/postprocessing/template_similarity.py:345: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  overlapping_ids = overlapping_j_list[i]


spike_amplitudes (workers: 36 processes fork):   0%|          | 0/8723 [00:00<?, ?it/s]


# Quality Metrics Tutorial

After spike sorting, you might want to validate the 'goodness' of the sorted units. This can be done using the
:code:`qualitymetrics` submodule, which computes several quality metrics of the sorted units.


In [ ]:
import spikeinterface.core as si
from spikeinterface.metrics import (
    compute_snrs,
    compute_presence_ratios,
    compute_isi_violations,
)

First, let's generate a simulated recording and sorting



In [ ]:
# sorting = sorting_TDC
# sorting_analyzer = sorting_TDC.to_shared_memory_sorting()

sorting_analyzer = si.create_sorting_analyzer(sorting=sorting, recording=recording_filtered)
sorting_analyzer.compute(['noise_levels','random_spikes','waveforms','templates','spike_locations','spike_amplitudes','correlograms','principal_components','quality_metrics','template_metrics'])


In [ ]:
sorting_analyzer.compute('template_metrics', include_multi_channel_metrics=True) 

# recording, sorting = si.generate_ground_truth_recording()
# print(recording)
# print(sorting)

In [12]:
all_metric_names = list(sorting_analyzer.get_extension('quality_metrics').get_data().keys()) + list(sorting_analyzer.get_extension('template_metrics').get_data().keys())
print(set(model.feature_names_in_).issubset(set(all_metric_names)))

AttributeError: 'NoneType' object has no attribute 'get_data'

## Create SortingAnalyzer

For quality metrics we need first to create a :code:`SortingAnalyzer`.



The :code:`spikeinterface.qualitymetrics` submodule has a set of functions that allow users to compute
metrics in a compact and easy way. To compute a single metric, one can simply run one of the
quality metric functions as shown below. Each function has a variety of adjustable parameters that can be tuned.



In [ ]:
## INPUTS: sorting_analyzer
presence_ratios = compute_presence_ratios(sorting_analyzer)
print(presence_ratios)
isi_violation_ratio, isi_violations_count = compute_isi_violations(sorting_analyzer)
print(isi_violation_ratio)
snrs = compute_snrs(sorting_analyzer)
print(snrs)

To compute more than one metric at once, we can use the :code:`SortingAnalyzer.compute("quality_metrics")`
function and indicate which metrics we want to compute. Then we can retrieve the results using the :code:`get_data()`
method as a ``pandas.DataFrame``.



In [ ]:
metrics_ext = sorting_analyzer.compute(
    "quality_metrics",
    metric_names=["presence_ratio", "snr", "amplitude_cutoff"],
    metric_params={
        "presence_ratio": {"bin_duration_s": 2.0},
    }
)
metrics = metrics_ext.get_data()
print(metrics)

Some metrics are based on the principal component scores, so the extension
must be computed before. For instance:



In [ ]:
sorting_analyzer.compute("principal_components", n_components=5, mode="by_channel_global", whiten=True)

metrics_ext = sorting_analyzer.compute(
    "quality_metrics",
    metric_names=[
        "mahalanobis",
        "d_prime",
    ],
)
metrics = metrics_ext.get_data()
print(metrics)

# Trained Models

In [ ]:
from spikeinterface.curation import model_based_label_units

labels_and_probabilities = model_based_label_units(
    sorting_analyzer = sorting_analyzer,
    repo_id = "SpikeInterface/toy_tetrode_model",
    trust_model = True
)

In [33]:
GOOD_UNIT_STRATEGY: str = "sua_high_conf"
PROB_THRESHOLD_HIGH: float = 0.8
curation = CurationConfig(strategy="sua_high_conf", prob_high=PROB_THRESHOLD_HIGH, analyzer_overwrite="always", require_cluster_info=False)


In [ ]:
result = run_phy_curation_pipeline(session, curation, patch_pandas_compat=True)
sorting = result.sorting
sorting_analyzer = result.sorting_analyzer
all_labels = result.all_labels
comparison = result.comparison
good_units = result.good_units
sorting_curated = result.sorting_curated
print(all_labels["prediction"].value_counts())
print(comparison.groupby(["phy_label", "prediction"]).size())


In [ ]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [ ]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [ ]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [34]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [35]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [36]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [37]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [38]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [39]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [40]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [41]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [42]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).


In [ ]:
sorting

In [ ]:
metrics_df = sorting_analyzer.get_extension("quality_metrics").get_data()
template_df = sorting_analyzer.get_extension("template_metrics").get_data()

out_dir = sorting_analyzer_folder / "exports"
out_dir.mkdir(exist_ok=True)
metrics_df.to_csv(out_dir / "quality_metrics.csv")
template_df.to_csv(out_dir / "template_metrics.csv")

In [ ]:
metrics_df

In [ ]:
template_df

## Write out labels to Phy

In [ ]:
# Handled by spikeinterface_pipeline.run_phy_curation_pipeline (cell 37).



/nfs/turbo/umms-kdiba/Bapun/RatS/Day1Openfield/spyk-circ
/nfs/turbo/umms-kdiba/Bapun/RatS/Day4Openfield/spyk-circ
/nfs/turbo/umms-kdiba/Bapun/RatS/Day5TwoNovel/spyk-circ


In [14]:
qm = sorting_analyzer.get_extension("quality_metrics").get_data()
print(qm.isna().sum())   # <-- THE key check: which metrics are NaN for every unit?
print(qm.describe())     # <-- real distribution of snr / isi / contamination

snr            0
firing_rate    1
dtype: int64
              snr  firing_rate
count  258.000000   257.000000
mean     6.646348     4.666586
std      6.493993     8.167132
min      0.000000     0.019145
25%      3.281160     0.427506
50%      4.465928     1.928190
75%      8.068428     5.454741
max     56.735449    77.622517


In [16]:
from spikeinterface.curation import threshold_metrics_label_units

labels = threshold_metrics_label_units(sorting_analyzer, thresholds={"isi_violations_ratio": {"less": 0.5}, "rp_contamination": {"less": 0.3}, "presence_ratio": {"greater": 0.8}}, pass_label="good", fail_label="mua", column_name="auto_label")
print(labels["auto_label"].value_counts())

ValueError: Only pd.DataFrame is supported for metrics.

In [32]:
from spikeinterface.curation import bombcell_label_units, bombcell_get_default_thresholds

labels = bombcell_label_units(sorting_analyzer=sorting_analyzer) # , thresholds=bombcell_get_default_thresholds()

ValueError: Metric(s) ['amplitude_median', 'amplitude_cutoff', 'num_spikes', 'rp_contamination', 'presence_ratio', 'drift_ptp'] specified in thresholds are not present in the quality metrics DataFrame. Available metrics are: ['snr', 'firing_rate', 'peak_to_trough_duration', 'trough_half_width', 'peak_half_width', 'repolarization_slope', 'recovery_slope', 'num_positive_peaks', 'num_negative_peaks', 'main_to_next_extremum_duration', 'peak_before_to_trough_ratio', 'peak_after_to_trough_ratio', 'peak_before_to_peak_after_ratio', 'main_peak_to_trough_ratio', 'trough_width', 'peak_before_width', 'peak_after_width', 'waveform_baseline_flatness', 'velocity_above', 'velocity_below', 'exp_decay', 'spread']

In [26]:
import spikeinterface as si

si.set_global_job_kwargs(n_jobs=11, chunk_duration="1s", progress_bar=True)

In [27]:
import spikeinterface.qualitymetrics as sqm

sorting_analyzer.compute({"spike_amplitudes": {}, "spike_locations": {}})


Compute : spike_amplitudes + spike_locations (workers: 11 processes fork):   0%|          | 0/8723 [00:00<?, ?…

/nfs/turbo/umms-kdiba/Bapun/RatS/spikeinterface/src/spikeinterface/sortingcomponents/peak_localization/center_of_mass.py:64: RuntimeWarning: invalid value encountered in divide
  coms = np.dot(wf_data, local_contact_locations) / (np.sum(wf_data, axis=1)[:, np.newaxis])
/nfs/turbo/umms-kdiba/Bapun/RatS/spikeinterface/src/spikeinterface/sortingcomponents/peak_localization/center_of_mass.py:64: RuntimeWarning: invalid value encountered in divide
  coms = np.dot(wf_data, local_contact_locations) / (np.sum(wf_data, axis=1)[:, np.newaxis])
/nfs/turbo/umms-kdiba/Bapun/RatS/spikeinterface/src/spikeinterface/sortingcomponents/peak_localization/center_of_mass.py:64: RuntimeWarning: invalid value encountered in divide
  coms = np.dot(wf_data, local_contact_locations) / (np.sum(wf_data, axis=1)[:, np.newaxis])
/nfs/turbo/umms-kdiba/Bapun/RatS/spikeinterface/src/spikeinterface/sortingcomponents/peak_localization/center_of_mass.py:64: RuntimeWarning: invalid value encountered in divide
  coms = np.d

ValueError: Metric rp_contamination not in available metrics ['num_spikes', 'firing_rate', 'presence_ratio', 'snr', 'isi_violation', 'rp_violation', 'sliding_rp_violation', 'synchrony', 'firing_range', 'amplitude_cv', 'amplitude_cutoff', 'noise_cutoff', 'amplitude_median', 'drift', 'sd_ratio', 'mahalanobis', 'd_prime', 'nearest_neighbor', 'silhouette', 'nn_advanced']

In [ ]:
sorting_analyzer.compute("quality_metrics", metric_names=["snr", "firing_rate", "num_spikes", "rp_contamination", "presence_ratio", "amplitude_median", "amplitude_cutoff", "drift_ptp", "isi_violations_ratio"])


In [29]:
qm = sorting_analyzer.get_extension("quality_metrics").get_data()
print(qm.columns.tolist())   # confirm all six missing names now appear
print(qm.isna().sum())       # see which (if any) come back NaN

['snr', 'firing_rate']
snr            0
firing_rate    1
dtype: int64


In [21]:
# comp_names = ['num_positive_peaks', 'num_negative_peaks', 'peak_to_trough_duration', 'waveform_baseline_flatness', 'peak_after_to_trough_ratio', 'exp_decay']
# extensions_to_run = {k:{} for k in comp_names}

extensions_to_run = {
    "random_spikes": {},
    "waveforms": {},
    "templates": {},
    "noise_levels": {},
    "spike_amplitudes": {},
    "template_metrics": {"include_multi_channel_metrics": True},
    "quality_metrics": {},
}

# print(sorting_analyzer.get_extension("template_metrics").get_data().columns.tolist())
# print(sorting_analyzer.get_extension("quality_metrics").get_data().columns.tolist())



# Iterate through and compute only if the extension is missing
for ext_name, ext_params in extensions_to_run.items():
    if not sorting_analyzer.has_extension(ext_name):
        print(f"Computing missing extension: {ext_name}")
        sorting_analyzer.compute(ext_name, **ext_params)
    else:
        print(f"Extension '{ext_name}' already exists. Skipping.")

# If you computed new extensions and your analyzer is memory-backed, 
# remember to save it afterwards:
# analyzer.save_as(format="zarr", folder="/path/to/new_analyzer.zarr")

Extension 'random_spikes' already exists. Skipping.
Extension 'waveforms' already exists. Skipping.
Extension 'templates' already exists. Skipping.
Extension 'noise_levels' already exists. Skipping.
Extension 'spike_amplitudes' already exists. Skipping.
Computing missing extension: template_metrics


/nfs/turbo/umms-kdiba/Bapun/RatS/spikeinterface/src/spikeinterface/metrics/template/template_metrics.py:304: UserWarning: With less than 10 channels, multi-channel metrics might not be reliable.
  warnings.warn(


Extension 'quality_metrics' already exists. Skipping.


/nfs/turbo/umms-kdiba/Bapun/RatS/spikeinterface/src/spikeinterface/metrics/template/metrics.py:999: RuntimeWarning: invalid value encountered in divide
  MM = MM / np.max(MM)
/nfs/turbo/umms-kdiba/Bapun/RatS/spikeinterface/src/spikeinterface/core/analyzer_extension_core.py:1283: UserWarning: Error computing metric spread: zero-size array to reduction operation maximum which has no identity
  warnings.warn(f"Error computing metric {metric_name}: {e}")


In [30]:
print(sorting_analyzer.get_extension("template_metrics").get_data().columns.tolist())
print(sorting_analyzer.get_extension("quality_metrics").get_data().columns.tolist())


['peak_to_trough_duration', 'trough_half_width', 'peak_half_width', 'repolarization_slope', 'recovery_slope', 'num_positive_peaks', 'num_negative_peaks', 'main_to_next_extremum_duration', 'peak_before_to_trough_ratio', 'peak_after_to_trough_ratio', 'peak_before_to_peak_after_ratio', 'main_peak_to_trough_ratio', 'trough_width', 'peak_before_width', 'peak_after_width', 'waveform_baseline_flatness', 'velocity_above', 'velocity_below', 'exp_decay', 'spread']
['snr', 'firing_rate']
